# Sprint 10 - Webinar 31 · Diseño de ETL en Python (con SQLite) · Student Version

**Proyecto de la sesión:** construir un ETL sencillo (Extract → Transform → Load) que toma datos *crudos* de una tienda, los limpia y los carga a una base de datos SQLite.

> Nivel: **principiante**. Código simple, paso a paso, sin trucos.

### ¿Cómo usar este cuaderno?
- Lee el contexto y toma apuntes en los espacios marcados.
- Completa las celdas con `TODO`.
- Ejecuta por bloques y valida cada resultado antes de seguir.
- Si algo falla, documenta el error y qué hiciste para corregirlo.

## Fecha
> Completa aquí la fecha de la sesión.

**Fecha:** ______________________________________

**Nombre del estudiante:** ______________________________________

## Objetivo de la sesión práctica (100 min)
Al finalizar esta clase, podrás:

1. Explicar qué es **ETL** (y cuándo usarlo).
2. Extraer datos desde archivos **CSV** y **JSON** usando `pandas`.
3. Aplicar transformaciones básicas (limpieza, tipos, validaciones, joins).
4. Cargar datos a una base **SQLite** y verificar que todo quedó bien.
5. Armar un **pipeline** simple (funciones) para repetir el proceso.


## Agenda sugerida (100 minutos)
1. Contexto + checklist de entregables (5 min)
2. **Actividad 0 (breakout rooms):** diseñar el ETL en papel (10 min)
3. Ejercicio 1: crear datos crudos + estructura del proyecto (15 min)
4. Ejercicio 2: **Extract** (CSV + JSON) + exploración rápida (15 min)
5. Ejercicio 3: **Transform** (limpieza + modelo final) (20 min)
6. Ejercicio 4: **Load** a SQLite + consultas de verificación (20 min)
7. Ejercicio 5: mini-orquestación (pipeline) + re-ejecución (10 min)
8. Takeaways + cierre (5 min)


## Contexto
Imagina que trabajas como Data Analyst en una tienda online. Cada día recibes datos desde diferentes fuentes:

- Un archivo `customers_raw.csv` exportado desde un CRM.
- Un archivo `orders_raw.csv` exportado desde el sistema de ventas.
- Un archivo `products_raw.json` con el catálogo de productos.

El problema: **los datos vienen sucios** (espacios, mayúsculas, tipos incorrectos, fechas raras, valores faltantes).

Tu objetivo hoy es diseñar un ETL para:

1) **Extract**: leer los archivos crudos tal como llegan.
2) **Transform**: limpiar, estandarizar y construir tablas listas para análisis.
3) **Load**: cargar el resultado a una base SQLite para consultas.

### ¿ETL vs ELT?
- **ETL**: transformas *antes* de cargar (común en pipelines tradicionales y cuando el destino es simple).
- **ELT**: cargas primero y transformas después (muy común en data warehouses modernos).

Hoy haremos **ETL** porque:
- SQLite es liviano (ideal para practicar),
- queremos entender bien qué significa “transformar”,
- y necesitamos un resultado listo para análisis.


## Checklist de entregables (al final de la clase)
- [ ] Carpeta `data/raw/` con archivos crudos (CSV/JSON).
- [ ] Carpeta `data/processed/` con un dataset limpio (CSV).
- [ ] Base `data/db/tienda_etl.db` con tablas cargadas.
- [ ] 2–3 consultas SQL de verificación (conteos, joins).
- [ ] Función `run_etl()` que ejecute el pipeline completo.


---

# Actividad 0 · Calentamiento (10 min)

Antes de tocar código, diseñemos el ETL en equipo.

### Instrucciones (breakout rooms, 3–4 personas)
1) Lean el objetivo: **“queremos analizar ventas por cliente y por producto”**.
2) Respondan:
   - ¿Qué tablas finales (destino) necesitamos? (pista: *dimensiones* y *hechos*)
   - ¿Qué columnas son claves? (pista: `customer_id`, `order_id`, `product_id`)
   - ¿Qué problemas de calidad podrían venir en los datos crudos?
   - ¿Qué transformaciones mínimas harías?
3) Escriban una propuesta corta (5–8 líneas).

### Pista
Un modelo típico para análisis de ventas:
- `dim_customers` (una fila por cliente)
- `dim_products` (una fila por producto)
- `fact_orders` (una fila por orden o por ítem)

### Solución ejemplo (respuesta esperada)
- Tablas destino: `dim_customers`, `dim_products`, `fact_orders`.
- Claves: `customer_id`, `product_id`, `order_id`.
- Problemas: duplicados, emails con mayúsculas/espacios, fechas como texto, precios con símbolos, nulos.
- Transformaciones: renombrar columnas, `strip()` espacios, convertir fechas a `datetime`, convertir precios a número, validar rangos y eliminar/ajustar registros inválidos.


### Espacio para propuesta del equipo
- Tablas finales: 
- Claves principales: 
- Problemas de calidad esperados: 
- Transformaciones mínimas: 


In [ ]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================

import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


---

# Ejercicio 1 · Crear datos crudos + estructura del proyecto (15 min)

**Meta:** crear una carpeta de proyecto reproducible y generar archivos crudos con *errores típicos*.

### ¿Por qué separamos `raw/` y `processed/`?
- `raw/`: datos originales **sin modificar** (si algo sale mal, puedes volver a empezar)
- `processed/`: datos listos para análisis (resultado del ETL)
- `db/`: base de datos con tablas cargadas

### Estructura que construiremos
```
data/
  raw/
  processed/
  db/
```

### Tu turno
1) Crea las carpetas.
2) Genera 3 archivos:
   - `customers_raw.csv`
   - `orders_raw.csv`
   - `products_raw.json`

Los datos pueden ser sintéticos (inventados), pero deben incluir algunos problemas: espacios, mayúsculas, valores faltantes, fechas como texto, etc.

### Pista
- Usa `Path(...).mkdir(parents=True, exist_ok=True)`.
- Puedes crear `DataFrame`s y luego `to_csv(...)`.
- Para JSON, puedes guardar una lista de diccionarios.


### Antes de programar
- ¿Qué carpetas vas a crear? 
- ¿Qué errores intencionales incluirás en los datos crudos? 
- ¿Cómo verificarás que los archivos sí quedaron guardados? 


In [ ]:
# Tu turno: crea carpetas y genera los archivos crudos (raw)
# TIP: si algo falla, vuelve a ejecutar la celda: debe ser segura (idempotente).

# 1) Crea carpetas: data/raw, data/processed, data/db
# 2) Crea dataframes para customers y orders
# 3) Guarda customers_raw.csv y orders_raw.csv
# 4) Crea products_raw.json (lista de productos)

# Escribe tu solución aquí 👇


In [ ]:
# ============================================================
# Desarrollo guiado · Ejercicio 1
# ============================================================

# Objetivo:
# 1) Crear carpetas: data/raw, data/processed, data/db
# 2) Generar datasets crudos con errores típicos
# 3) Guardarlos en CSV / JSON

from pathlib import Path

# TODO 1: define las rutas de trabajo
RAW_DIR = ...
PROCESSED_DIR = ...
DB_DIR = ...

# TODO 2: crea las carpetas de forma segura
# Pista: usa mkdir(parents=True, exist_ok=True)

# Escribe tu código aquí 👇


# TODO 3: crea un dataframe de customers con errores intencionales
# Sugerencias de columnas:
# - Customer ID
# - Name
# - Email
# - City
# Incluye algunos problemas como espacios extra, mayúsculas, nulos o duplicados

customers = ...

# TODO 4: crea un dataframe de orders con errores intencionales
# Sugerencias de columnas:
# - order_id
# - customer_id
# - product_id
# - quantity
# - unit_price
# - order_date

orders = ...

# TODO 5: crea una lista de productos y guárdala como JSON
products = ...

# TODO 6: guarda los archivos en RAW_DIR
# - customers_raw.csv
# - orders_raw.csv
# - products_raw.json

# Verificación mínima sugerida:
# print(list(RAW_DIR.iterdir()))

---

# Ejercicio 2 · Extract (CSV + JSON) + exploración rápida (15 min)

**Meta:** leer datos crudos y entender su forma (filas/columnas, tipos, nulos).

### Fundamento teórico (muy corto)
- En **Extract**, tu prioridad es *no perder información*. Si el dato viene como texto raro, lo lees como texto y lo transformas después.
- Las primeras preguntas siempre son:
  1) ¿Cuántas filas/columnas tengo?
  2) ¿Qué tipos detectó Python?
  3) ¿Qué tan sucios están? (nulos, duplicados, valores fuera de rango)

### Tu turno
1) Carga `customers_raw.csv` y `orders_raw.csv` con pandas.
2) Carga `products_raw.json`.
3) Muestra: `head()`, `shape`, `info()` y conteo de nulos por columna.

### Pista
- CSV: `pd.read_csv(ruta)`
- JSON: `json.load(open(...))` y luego `pd.DataFrame(lista)`


### Registro del estudiante
- ¿Qué dataset parece más sucio y por qué? 
- ¿Qué columnas sospechas que requerirán limpieza de tipos? 
- ¿Qué validaciones mínimas harías antes de transformar? 


In [ ]:
# Tu turno: carga y explora los datos crudos

# 1) Carga customers_raw.csv y orders_raw.csv
# 2) Carga products_raw.json
# 3) Explora: head, shape, info, nulos por columna

# Escribe tu solución aquí 👇


In [ ]:
# ============================================================
# Desarrollo guiado · Ejercicio 2
# ============================================================

# TODO 1: carga los archivos crudos
customers_raw = ...
orders_raw = ...

# TODO 2: abre el JSON y conviértelo a DataFrame
# Pista:
# with open(RAW_DIR / "products_raw.json", "r", encoding="utf-8") as f:
#     products_list = json.load(f)
# products_raw = pd.DataFrame(products_list)

products_raw = ...

# TODO 3: explora cada dataset
# Revisa como mínimo:
# - .head()
# - .shape
# - .info()
# - nulos por columna
# - duplicados (si aplica)

# customers_raw
display(...)
print(...)
# customers_raw.info()
display(...)

# orders_raw
display(...)
print(...)
# orders_raw.info()
display(...)

# products_raw
display(...)
print(...)
# products_raw.info()
display(...)

# Registro breve:
# ¿Qué problema encontraste primero?
# Respuesta: ________________________________________________

---

# Ejercicio 3 · Transform (limpieza + modelo final) (20 min)

**Meta:** convertir datos crudos en tablas limpias para análisis.

### Fundamento teórico
Transform es donde definimos **reglas de negocio** y **reglas de calidad**. Ejemplos:
- Estandarizar nombres de columnas (snake_case)
- Limpiar strings (`strip`, `lower`, `title`)
- Convertir tipos (`to_datetime`, `to_numeric`)
- Eliminar/ajustar registros inválidos (ej. `quantity <= 0`)
- Crear columnas derivadas (ej. `total = quantity * unit_price`)

### Tu turno
1) Limpia `customers_raw` para crear `dim_customers`.
2) Limpia `products_raw` para crear `dim_products`.
3) Limpia `orders_raw` para crear `fact_orders`.
4) Guarda un CSV limpio en `data/processed/orders_clean.csv`.

### Pista
- Para renombrar columnas: `df.rename(columns={...})`
- Para limpiar texto: `df['col'].astype(str).str.strip().str.lower()`
- Para números: `pd.to_numeric(..., errors='coerce')`
- Para fechas: `pd.to_datetime(..., errors='coerce', dayfirst=True)`


### Plan de transformación
- Reglas para `dim_customers`: 
- Reglas para `dim_products`: 
- Reglas para `fact_orders`: 
- Validaciones clave antes de cargar a SQLite: 


In [ ]:
# Tu turno: crea dim_customers, dim_products y fact_orders

# Reglas mínimas sugeridas:
# - Columnas en snake_case
# - Emails en minúscula y sin espacios
# - City con strip y formato tipo título (Title Case)
# - unit_price numérico (float)
# - order_date a datetime
# - quantity a entero; elimina quantity <= 0 o nulos
# - total = quantity * unit_price
# - Elimina duplicados en customers por customer_id

# Escribe tu solución aquí 👇


In [ ]:
# ============================================================
# Desarrollo guiado · Ejercicio 3
# ============================================================

# Meta:
# construir dim_customers, dim_products y fact_orders

# Sugerencia: puedes crear una función auxiliar para normalizar nombres de columnas.
def to_snake_case(columns):
    # TODO: devuelve una lista con nombres en snake_case
    # Ejemplo: "Customer ID" -> "customer_id"
    return ...


# -------------------------
# 1) dim_customers
# -------------------------
# Reglas sugeridas:
# - renombrar columnas
# - trim en strings
# - name en Title Case
# - email en minúsculas
# - eliminar duplicados por customer_id

dim_customers = customers_raw.copy()

# TODO: aplica las reglas
# Escribe tu código aquí 👇


# Verificación sugerida
display(dim_customers.head())
display(dim_customers.isna().sum())


# -------------------------
# 2) dim_products
# -------------------------
# Reglas sugeridas:
# - product_id limpio
# - product_name en formato consistente
# - category normalizada
# - unit_price como número

dim_products = products_raw.copy()

# TODO: aplica las reglas
# Escribe tu código aquí 👇


display(dim_products.head())
display(dim_products.dtypes)


# -------------------------
# 3) fact_orders
# -------------------------
# Reglas sugeridas:
# - order_date a datetime
# - quantity a entero o numérico
# - unit_price a float
# - eliminar quantity <= 0 o nulos
# - crear total = quantity * unit_price

fact_orders = orders_raw.copy()

# TODO: aplica las reglas
# Escribe tu código aquí 👇


display(fact_orders.head())
display(fact_orders.describe(include="all"))

# -------------------------
# 4) Guardado de procesados
# -------------------------
# TODO: guarda al menos un archivo limpio en data/processed/
# Ejemplo:
# fact_orders.to_csv(PROCESSED_DIR / "fact_orders_clean.csv", index=False)

# Reflexión:
# ¿Qué regla de calidad fue la más importante?
# Respuesta: ________________________________________________

---

# Ejercicio 4 · Load a SQLite + consultas de verificación (20 min)

**Meta:** cargar `dim_customers`, `dim_products` y `fact_orders` a SQLite.

### Fundamento teórico
Una base SQL te permite:
- almacenar datos con estructura y reglas (tipos, llaves)
- hacer consultas reproducibles (`GROUP BY`, `JOIN`, etc.)
- compartir resultados con otras herramientas

SQLite es ideal para practicar porque:
- no requiere instalar un servidor
- es un archivo `.db`

### Tu turno
1) Crea una base `data/db/tienda_etl.db`.
2) Carga las 3 tablas.
3) Haz 3 consultas de verificación:
   - Conteo de filas por tabla
   - Total de ventas (sumatoria de `total`)
   - Top 5 productos por ventas

### Pista
- Conexión: `sqlite3.connect('ruta.db')`
- Cargar con pandas: `df.to_sql('tabla', conn, if_exists='replace', index=False)`
- Consultar: `pd.read_sql_query('SELECT ...', conn)`


### Registro de verificación
- ¿Cuántas filas esperas por tabla? 
- ¿Qué consulta usarás para revisar ventas totales? 
- ¿Cómo comprobarás que los joins funcionan? 


In [ ]:
# Tu turno: crea la base y carga las tablas

# 1) Conecta/crea la base de datos
# 2) Carga dim_customers, dim_products, fact_orders
# 3) Corre consultas de verificación

# Escribe tu solución aquí 👇


In [ ]:
# ============================================================
# Desarrollo guiado · Ejercicio 4
# ============================================================

# TODO 1: crea o conecta la base de datos SQLite
db_path = ...
conn = ...

# TODO 2: carga las tablas con to_sql()
# - dim_customers
# - dim_products
# - fact_orders

# Escribe tu código aquí 👇


# TODO 3: crea una consulta de conteo por tabla
q_counts = '''
-- Escribe aquí tu SQL
'''

# TODO 4: crea una consulta para total de ventas
q_total_sales = '''
-- Escribe aquí tu SQL
'''

# TODO 5: crea una consulta para top 5 productos por ventas
q_top_products = '''
-- Escribe aquí tu SQL
'''

# Ejecuta y muestra los resultados
display(pd.read_sql_query(q_counts, conn))
display(pd.read_sql_query(q_total_sales, conn))
display(pd.read_sql_query(q_top_products, conn))

# Pregunta guía:
# ¿Qué evidencia demuestra que la carga quedó bien?
# Respuesta: ________________________________________________

---

# Ejercicio 5 · Mini-orquestación (pipeline) + re-ejecución (10 min)

**Meta:** organizar el ETL en funciones para que sea repetible.

### Fundamento teórico
Un buen ETL (aunque sea pequeño) debe ser:
- **repetible** (puedes correrlo 10 veces y siempre funciona)
- **legible** (otra persona entiende qué hace)
- **validable** (tiene chequeos mínimos)

### Tu turno
1) Crea 3 funciones: `extract()`, `transform()`, `load()`.
2) Crea `run_etl()` que ejecute todo.
3) Ejecuta `run_etl()` y verifica que crea el `.db`.

### Pista
- `extract()` debería devolver los dataframes crudos.
- `transform()` debería recibir crudos y devolver limpios.
- `load()` debería recibir limpios y escribir en SQLite.


### Diseño del pipeline
- ¿Qué debe devolver `extract()`? 
- ¿Qué debe recibir y devolver `transform()`? 
- ¿Qué validaciones mínimas pondrías en `load()`? 


In [ ]:
# Tu turno: organiza el ETL en funciones y ejecuta run_etl()

# Escribe tu solución aquí 👇


In [ ]:
# ============================================================
# Desarrollo guiado · Ejercicio 5
# ============================================================

# Objetivo:
# organizar el ETL en funciones y ejecutar run_etl()

def extract(raw_dir: Path):
    """Lee archivos crudos y devuelve los dataframes originales."""
    # TODO
    customers_raw = ...
    orders_raw = ...
    products_raw = ...
    return customers_raw, orders_raw, products_raw


def transform(customers_raw: pd.DataFrame, orders_raw: pd.DataFrame, products_raw: pd.DataFrame):
    """Limpia y construye dim_customers, dim_products y fact_orders."""
    # TODO
    dim_customers = ...
    dim_products = ...
    fact_orders = ...
    return dim_customers, dim_products, fact_orders


def load(dim_customers: pd.DataFrame, dim_products: pd.DataFrame, fact_orders: pd.DataFrame, db_path: Path):
    """Carga tablas limpias a SQLite."""
    # TODO
    return ...


def run_etl(raw_dir: Path, db_path: Path):
    """Ejecuta el flujo completo Extract -> Transform -> Load."""
    # TODO
    # 1) extraer
    # 2) transformar
    # 3) cargar
    # 4) imprimir mensaje final o devolver objetos
    return ...


# Ejecuta tu pipeline aquí 👇
# run_etl(RAW_DIR, DB_DIR / "tienda_etl.db")

# Auto-chequeo:
# - ¿La base se creó correctamente?
# - ¿Puedes volver a correr la función sin que falle?

## Takeaways de la sesión práctica

Completa al finalizar la clase:

- ETL significa: ________________________________________________
- La etapa más delicada en este ejercicio fue: ____________________
- Una validación que no debería faltar en este pipeline: __________
- La diferencia más importante entre **raw** y **processed** es: __
- SQLite me sirve en este contexto porque: ________________________

### Autoevaluación rápida
Marca con una X:
- [ ] Puedo explicar ETL con mis palabras.
- [ ] Puedo leer archivos CSV y JSON con pandas.
- [ ] Puedo limpiar columnas y tipos básicos.
- [ ] Puedo cargar datos a SQLite.
- [ ] Puedo organizar el flujo en funciones.

## Cierre

**Reflexión final**
1. ¿Qué entiendes por **ETL**?  
   Respuesta: ________________________________________________

2. ¿En qué casos usarías **ELT** en vez de ETL?  
   Respuesta: ________________________________________________

3. ¿Qué cuidados o recomendaciones tendrías en cuenta al construir un ETL real?  
   Respuesta: ________________________________________________

4. ¿Qué error apareció durante la práctica y cómo lo resolviste?  
   Respuesta: ________________________________________________

### Dudas pendientes
- ____________________________________________________________
- ____________________________________________________________
- ____________________________________________________________

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Extensión opcional
1. Agrega una validación extra, por ejemplo: verificar que cada `product_id` de `fact_orders` exista en `dim_products`.
2. Crea una tabla `ventas_por_dia` con SQL usando `GROUP BY date(order_date)`.
3. Documenta en una celda aparte cómo manejarías:
   - errores de lectura de archivos,
   - columnas faltantes,
   - duplicados inesperados,
   - re-ejecución diaria del pipeline.